In [0]:
/*
Project: Adventure Works Analytics
Purpose: Source Primary Key Profiling
Platform: Databricks SQL
Catalog: fea_academy
Schema: adventure_works
*/

-- ===========================================
-- Primary Key Profiling
-- ===========================================

--Obs.: define the columns names
SELECT
    'person_address' AS table_name,
    COUNT(*) AS total_records,
    COUNT(addressid) AS non_null_keys,
    COUNT(DISTINCT addressid) AS distinct_keys
FROM `fea_academy`.`adventure_works`.`person_address`

UNION ALL

SELECT
    'person_countryregion',
    COUNT(*),
    COUNT(countryregioncode),
    COUNT(DISTINCT countryregioncode)
FROM `fea_academy`.`adventure_works`.`person_countryregion`

UNION ALL

SELECT
    'person_person',
    COUNT(*),
    COUNT(businessentityid),
    COUNT(DISTINCT businessentityid)
FROM `fea_academy`.`adventure_works`.`person_person`

UNION ALL

SELECT
    'person_stateprovince',
    COUNT(*),
    COUNT(stateprovinceid),
    COUNT(DISTINCT stateprovinceid)
FROM `fea_academy`.`adventure_works`.`person_stateprovince`

UNION ALL

SELECT
    'production_product',
    COUNT(*),
    COUNT(productid),
    COUNT(DISTINCT productid)
FROM `fea_academy`.`adventure_works`.`production_product`

UNION ALL

SELECT
    'sales_creditcard',
    COUNT(*),
    COUNT(creditcardid),
    COUNT(DISTINCT creditcardid)
FROM `fea_academy`.`adventure_works`.`sales_creditcard`

UNION ALL

SELECT
    'sales_customer',
    COUNT(*),
    COUNT(customerid),
    COUNT(DISTINCT customerid)
FROM `fea_academy`.`adventure_works`.`sales_customer`

UNION ALL

SELECT
    'sales_salesorderheader',
    COUNT(*),
    COUNT(salesorderid),
    COUNT(DISTINCT salesorderid)
FROM `fea_academy`.`adventure_works`.`sales_salesorderheader`

UNION ALL

SELECT
    'sales_salesreason',
    COUNT(*),
    COUNT(salesreasonid),
    COUNT(DISTINCT salesreasonid)
FROM `fea_academy`.`adventure_works`.`sales_salesreason`

ORDER BY table_name;

/*
Notes:
    - The initial profiling has already yielded an important finding: almost all candidate keys exhibit uniqueness and an absence of null values
    - Investigate a record with a null value in countryregioncode
*/

-- ===========================================
-- Investigate NULL Country Region Keys
-- ===========================================

SELECT *
FROM `fea_academy`.`adventure_works`.`person_countryregion`
WHERE countryregioncode IS NULL;

/*
Notes:
    - For Namibia -> Expected ISO 3166-1 alpha-2 country code: 'NA'
*/

-- ===========================================
-- Sales Order Detail Composite Key
-- ===========================================

SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT salesorderid) AS distinct_orders,
    COUNT(DISTINCT salesorderdetailid) AS distinct_detail_ids,
    COUNT(DISTINCT CONCAT(
        CAST(salesorderid AS STRING),
        '-',
        CAST(salesorderdetailid AS STRING)
    )) AS distinct_composite_keys
FROM `fea_academy`.`adventure_works`.`sales_salesorderdetail`;

-- ===========================================
-- Sales Order Reason Composite Key
-- ===========================================

SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT salesorderid) AS distinct_orders,
    COUNT(DISTINCT salesreasonid) AS distinct_reasons,
    COUNT(DISTINCT CONCAT(
        CAST(salesorderid AS STRING),
        '-',
        CAST(salesreasonid AS STRING)
    )) AS distinct_composite_keys
FROM `fea_academy`.`adventure_works`.`sales_salesorderheadersalesreason`;

-- ===========================================
-- NULL Values in Composite Keys
-- ===========================================

SELECT *
FROM `fea_academy`.`adventure_works`.`sales_salesorderdetail`
WHERE salesorderid IS NULL
   OR salesorderdetailid IS NULL;

SELECT *
FROM `fea_academy`.`adventure_works`.`sales_salesorderheadersalesreason`
WHERE salesorderid IS NULL
   OR salesreasonid IS NULL;


/*
Notes:
PRIMARY KEY PROFILING:
    - 8 out of 9 candidate keys presented no NULL or duplicate values
    - person_countryregion contains one NULL countryregioncode:
        - For Namibia -> Expected ISO 3166-1 alpha-2 country code: 'NA'
        - Proposed treatment -> Replace NULL with 'NA' in the transformation layer
    - The affected record corresponds to Namibia (modifieddate: 2008-04-30)
COMPOSITE KEY VALIDATION:
    - sales_salesorderdetail:
        - 121,317 records
        - salesorderid + salesorderdetailid: 121,317 distinct combinations
        - No NULL key values or duplicate combinations
        - salesorderdetailid is also unique in the current dataset
sales_salesorderheadersalesreason:
    - 27,647 records
    - 27,647 distinct combinations of salesorderid + salesreasonid
    - No NULL key values or duplicate combinations
    - 23,012 distinct orders are associated with 7 sales reasons
ADDITIONAL FINDINGS:
    - sales_salesreason contains 10 registered reasons, but only 7 are associated with the analyzed orders
    - Some identifier columns were ingested as DOUBLE instead of LONG
    - Several date columns were ingested as STRING
    - All columns were reported as nullable in Databricks metadata
MODELING CONSIDERATIONS:
    - Investigate the NULL countryregioncode before defining geographic relationships
    - Handle salesorderheadersalesreason as a bridge table to prevent duplicated sales metrics
    - Standardize identifier and date data types in the staging layer
    - Distinguish gross sales, discounted sales, subtotal, and total due
    - Validate referential integrity before implementing dbt models
*/